# tbats-jax — GPU panel benchmark

Fits N independent TBATS models in parallel on one fused `vmap` call. This notebook reproduces the A100 headline from the v0.1.0 release: **10,000 hourly TBATS models fitted in ~200 s.** R `forecast::tbats` sequential baseline: ~2 hours.

Package: `tbats-jax==0.1.0` • [PyPI](https://pypi.org/project/tbats-jax/) • [Codeberg](https://codeberg.org/mospak/tbats-jax) • [Mirror](https://github.com/m0spak/tbats-jax)

## What you need
- **Any CUDA GPU runtime** (T4, L4, A100, H100). T4 = free tier, A100 = Colab Pro.
- Runtime → Change runtime type → GPU → Save **before** running.

## Benchmark reference numbers

Measured on real runs. CPU row is Apple Silicon M-series single-core for scale.

| Device          | N=500   | N=1000  | N=5000   | N=10000 | Note                        |
|-----------------|---------|---------|----------|---------|-----------------------------|
| CPU (M-series)  | 98 s    | 196 s   | 980 s*   | 1960 s* | linear in N                 |
| T4 (free)       | 46 s    | 104 s   | 1068 s   | —       | HBM wall past N≈2000        |
| A100 (Pro)      | —       | 53 s    | 99 s     | 201 s   | **10×** over CPU here       |

<sup>* CPU extrapolated linearly from N=500 measurement</sup>

## 1. Install

In [ ]:
# Core + CUDA JAX. `tbats-jax` pulls in optimistix / optax / scipy as deps.
!pip install -q --upgrade "tbats-jax==0.1.0" "jax[cuda12]"

## 2. Verify accelerator

Expect `gpu` + a CUDA device. `rho method (auto)` should show `power` (spectral norm via power iteration — the GPU-friendly admissibility form).

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
print('backend :', jax.default_backend())
print('devices :', jax.devices())

from tbats_jax.admissibility import _default_method
print('rho method (auto):', _default_method())

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv 2>/dev/null | head -2

## 3. Benchmark helper

Self-contained: builds a panel via `tbats_jax.datasets.synthesize_two_seasonal`, runs `fit_panel`, reports compile + warm wall times + mean SSR.

In [ ]:
import time
import numpy as np
import jax.numpy as jnp

from tbats_jax import TBATSSpec, fit_panel
from tbats_jax.datasets import synthesize_two_seasonal
from tbats_jax.transforms import raw_to_natural
from tbats_jax.kernel import neg_log_likelihood


def run(N: int, T: int = 1500):
    spec = TBATSSpec(
        seasonal=((24.0, 3), (168.0, 5)),
        use_trend=True,
        use_damping=True,
    )
    panel = np.stack(
        [synthesize_two_seasonal(n=T, seed=s) for s in range(N)],
        axis=0,
    )
    print(f'\n=== N={N}  T={T}  state_dim={spec.state_dim}  n_params={spec.n_params} ===')
    t0 = time.perf_counter()
    thetas_raw, nlls, compile_t, warm_wall = fit_panel(panel, spec, max_steps=1000)
    jax.block_until_ready(thetas_raw)
    total = time.perf_counter() - t0

    # mean SSR over the first 20 series
    ssrs = []
    for i in range(min(N, 20)):
        th = np.asarray(raw_to_natural(jnp.asarray(thetas_raw[i]), spec))
        nll = float(neg_log_likelihood(jnp.asarray(panel[i]), jnp.asarray(th), spec))
        ssrs.append(np.exp(nll / T) * T)

    print(f'compile   : {compile_t:.1f}s')
    print(f'warm wall : {warm_wall:.2f}s')
    print(f'per-series: {warm_wall / N * 1000:.2f} ms')
    print(f'mean SSR  : {np.mean(ssrs):.2f}')
    return {
        'N': N,
        'T': T,
        'compile_time': compile_t,
        'warm_wall': warm_wall,
        'per_series_ms': warm_wall / N * 1000,
        'mean_ssr_20': float(np.mean(ssrs)),
    }

## 4. Baseline: N=1000

On A100 expect ~40–60 ms/series. On T4 ~100 ms/series. On CPU ~200 ms/series.

In [ ]:
r1000 = run(N=1000)

## 5. The regime T4 lost: N=5000

T4 takes 1068 s here (loses to CPU). A100 should be ~100 s (~10× CPU). Memory footprint at T=1500, N=5000, float64: ~1 GB y + ~4 GB autodiff intermediates — comfortably fits A100's 40 GB.

In [ ]:
r5000 = run(N=5000)

## 6. The headline: N=10000

Ten thousand TBATS models in one fused `vmap`. R `forecast::tbats` sequential cost: ~2 hours. CPU linear extrapolation: ~33 min. A100 target: **~3 min**. Memory footprint: ~10 GB on 40 GB A100.

In [ ]:
r10000 = run(N=10000)

## 7. Stretch (optional): N=20000

Memory ~20 GB of autodiff state, still within A100's 40 GB. Compile time doubles. Skip to save credits; N=10000 is the main result.

In [ ]:
try:
    r20000 = run(N=20000)
except Exception as e:
    r20000 = None
    print(f'N=20000 skipped: {e}')

## 8. Summary

In [ ]:
cpu_per_series_ms = 196.0   # Apple Silicon M-series CPU at T=1500
r_per_series_ms   = 343.0   # R forecast::tbats fixed-k at T=1500

print(f'accelerator: {jax.default_backend().upper()} — {jax.devices()[0]}\n')
hdr = f'{"N":>6} | {"compile":>9} | {"warm":>8} | {"ms/ser":>7} | {"CPU eq":>10} | {"R eq":>10} | {"vs CPU":>8} | {"vs R":>7}'
print(hdr)
print('-' * len(hdr))
rows = [(1000, r1000), (5000, r5000), (10000, r10000)]
if r20000 is not None:
    rows.append((20000, r20000))
for N, r in rows:
    cpu_eq = N * cpu_per_series_ms / 1000
    r_eq   = N * r_per_series_ms / 1000
    sp_cpu = cpu_eq / r['warm_wall']
    sp_r   = r_eq / r['warm_wall']
    print(f'{N:>6} | {r["compile_time"]:>8.1f}s | {r["warm_wall"]:>7.2f}s | {r["per_series_ms"]:>6.2f} | {cpu_eq:>9.1f}s | {r_eq:>9.1f}s | {sp_cpu:>7.2f}x | {sp_r:>6.1f}x')

## 9. Quick forecast demo (no benchmark overhead)

Single series, 30-day daily forecast. Run this even without a GPU — works on CPU too.

In [ ]:
from tbats_jax import fit_jax, forecast
from tbats_jax.datasets import synthesize_daily

y = synthesize_daily(n=730, seed=0)
spec = TBATSSpec(
    seasonal=((7.0, 3), (365.25, 5)),
    use_trend=True,
    use_damping=True,
)
res = fit_jax(y, spec, max_steps=500)
preds = np.asarray(forecast(y, jnp.asarray(res.theta), spec, horizon=30))
print(f'Fit: NLL={res.neg_log_lik_clean:.2f}  wall={res.wall_time:.2f}s')
print(f'30-day forecast, first 5 values: {preds[:5]}')
print(f'                 last 5 values: {preds[-5:]}')